In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Remplacez par le nom de votre dépôt sur le Hub
MODEL_ID = "facebook/nllb-200-distilled-600M" 

# Déterminer le périphérique (GPU si disponible, sinon CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Utilisation du périphérique : {device}")

# Charger le tokenizer et le modèle
print("Chargement du tokenizer et du modèle...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID).to(device) # <-- Envoyer le modèle sur le bon périphérique
print("Modèle et tokenizer chargés.")


def translate(text_to_translate, source_lang_code, target_lang_code):
    """
    Fonction de traduction qui gère la tokenization, la génération et le décodage.
    """
    # 1. Ajouter le code de la langue source au texte
    text_with_lang_code = f">>{source_lang_code}<< {text_to_translate}"
    
    # 2. Tokenizer le texte d'entrée
    inputs = tokenizer(text_with_lang_code, return_tensors="pt").to(device)
    
    # 3. Générer la traduction en forçant le début avec le code de la langue cible
    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(target_lang_code),
        max_new_tokens=256 # Définir une longueur maximale pour éviter les traductions trop longues
    )
    
    # 4. Décoder les tokens générés pour obtenir le texte
    translated_text = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
    
    return translated_text


# --- Exemple 1: Traduction Français -> Darija ---
text_fr = "Bonjour tout le monde. J'espère que vous allez bien."
print(f"\nTexte source (FR) : {text_fr}")
translation_dr = translate(text_fr, "fra_Latn", "ary_Arab")
print(f"Traduction (DR) : {translation_dr}")

# --- Exemple 2: Traduction Anglais -> Darija ---
text_en = "This is a test to check if the model is working correctly."
print(f"\nTexte source (EN) : {text_en}")
translation_dr_from_en = translate(text_en, "eng_Latn", "ary_Arab")
print(f"Traduction (DR) : {translation_dr_from_en}")

# --- Exemple 3: Traduction Darija -> Français ---
text_dr = "السلام عليكم، كيدايرين؟" 
print(f"\nTexte source (DR) : {text_dr}")
translation_fr = translate(text_dr, "ary_Arab", "fra_Latn")
print(f"Traduction (FR) : {translation_fr}")

Utilisation du périphérique : cuda
Chargement du tokenizer et du modèle...
Modèle et tokenizer chargés.

Texte source (FR) : Bonjour tout le monde. J'espère que vous allez bien.
Traduction (DR) : "هلا الجميع. أتمنى أن تكونوا بخير.

Texte source (EN) : This is a test to check if the model is working correctly.
Traduction (DR) : >> << هادي اختبار لتحقق من إن كان النموذج يعمل بشكل صحيح.

Texte source (DR) : السلام عليكم، كيدايرين؟
Traduction (FR) : Bonjour, vous voulez bien ?


In [7]:
from azure.cognitiveservices.speech import SpeechConfig, SpeechSynthesizer

# Configuration avec votre clé Azure (créez-la sur portal.azure.com)
speech_config = SpeechConfig(
    subscription="votre-clé-api-ici",  # Ex: "a1b2c3d4e5f6g7h8i9j0"
    region="eastus"                   # "eastus" ou "westeurope"
)

# Voix marocaine native (choisissez l'une ou l'autre)
speech_config.speech_synthesis_voice_name = "ar-MA-JamalNeural"  # Masculin
# speech_config.speech_synthesis_voice_name = "ar-MA-MounaNeural"  # Féminin

# Synthèse du texte
synthesizer = SpeechSynthesizer(speech_config)
result = synthesizer.speak_text_async("بغيت ناكول كسكس ف مرّاكش").get()

# Sauvegarde du fichier audio
with open("darija.wav", "wb") as audio_file:
    audio_file.write(result.audio_data)

print("✅ Fichier audio généré : darija.wav")

ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1342:(snd_func_refer) error evaluating name
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5727:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM default


✅ Fichier audio généré : darija.wav


In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
from evaluate import load
from tqdm import tqdm
from collections import defaultdict
from itertools import permutations

# --- CONFIGURATION ---
BASE_MODEL_ID = "facebook/nllb-200-distilled-600M"
TEST_DATA_FILE = "test_dataset.json"
NUM_SAMPLES_TO_TEST = 10  # <-- MODIFICATION : Nombre d'exemples à tester
# ---------------------

def evaluate_base_model_performance():
    """
    Charge le modèle de base NLLB, l'évalue sur un sous-ensemble du jeu de test
    et affiche les scores SacreBLEU pour chaque direction de traduction.
    """
    print(f"🚀 Évaluation du modèle de base : {BASE_MODEL_ID}")
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Utilisation du périphérique : {device.upper()}")

    print("...Chargement du modèle et du tokenizer (peut prendre un moment)...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
    model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL_ID).to(device)
    model.eval()
    print("✅ Modèle et tokenizer chargés.")
    
    print(f"📂 Chargement du jeu de test depuis : {TEST_DATA_FILE}")
    full_test_dataset = load_dataset("json", data_files=TEST_DATA_FILE, split="train")

    # --- MODIFICATION : Sélectionner un sous-ensemble du jeu de données ---
    if NUM_SAMPLES_TO_TEST is not None and len(full_test_dataset) > NUM_SAMPLES_TO_TEST:
        print(f"✂️  Sélection des {NUM_SAMPLES_TO_TEST} premiers exemples pour un test rapide...")
        test_dataset_subset = full_test_dataset.select(range(NUM_SAMPLES_TO_TEST))
    else:
        test_dataset_subset = full_test_dataset
    
    print("...Matérialisation du jeu de données en mémoire...")
    test_data_list = list(test_dataset_subset)
    print(f"✅ {len(test_data_list)} exemples chargés pour le test.")

    results = defaultdict(lambda: {"predictions": [], "references": []})
    
    print("\n⚙️  Démarrage de l'inférence sur le jeu de test...")
    with torch.no_grad():
        for example in tqdm(test_data_list, desc="Traduction des phrases"):
            translation_data = example.get('translation', {})
            
            valid_translations = {
                lang: text 
                for lang, text in translation_data.items() if text is not None and text.strip()
            }
            
            if len(valid_translations) < 2:
                continue

            for source_lang, target_lang in permutations(valid_translations.keys(), 2):
                source_text = valid_translations[source_lang]
                target_text = valid_translations[target_lang]

                input_text = f">>{source_lang}<< {source_text}"
                inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=1024).to(device)

                generated_tokens = model.generate(
                    **inputs,
                    forced_bos_token_id=tokenizer.convert_tokens_to_ids(target_lang),
                    max_new_tokens=256
                )
                
                prediction = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
                
                direction = f"{source_lang} -> {target_lang}"
                results[direction]["predictions"].append(prediction)
                results[direction]["references"].append([target_text])

    print("✅ Inférence terminée.")

    print("\n=======================================================")
    print(f"   SCORES BLEU DU MODÈLE DE BASE ({BASE_MODEL_ID}) sur {len(test_data_list)} exemples")
    print("=======================================================")
    
    bleu_metric = load("sacrebleu")
    
    for direction, data in results.items():
        if data["predictions"]:
            score = bleu_metric.compute(predictions=data["predictions"], references=data["references"])
            print(f"  - Direction {direction.ljust(25)}: BLEU = {score['score']:.2f}")

    print("=======================================================")


if __name__ == "__main__":
    evaluate_base_model_performance()

🚀 Évaluation du modèle de base : facebook/nllb-200-distilled-600M
Utilisation du périphérique : CUDA
...Chargement du modèle et du tokenizer (peut prendre un moment)...
✅ Modèle et tokenizer chargés.
📂 Chargement du jeu de test depuis : test_dataset.json
✂️  Sélection des 10 premiers exemples pour un test rapide...
...Matérialisation du jeu de données en mémoire...
✅ 10 exemples chargés pour le test.

⚙️  Démarrage de l'inférence sur le jeu de test...


Traduction des phrases: 100%|██████████| 10/10 [02:48<00:00, 16.80s/it]


✅ Inférence terminée.

   SCORES BLEU DU MODÈLE DE BASE (facebook/nllb-200-distilled-600M) sur 10 exemples
  - Direction ary_Arab -> eng_Latn     : BLEU = 12.54
  - Direction eng_Latn -> ary_Arab     : BLEU = 0.34
  - Direction fra_Latn -> ary_Arab     : BLEU = 3.86
  - Direction ary_Arab -> fra_Latn     : BLEU = 2.93


In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
from evaluate import load
from tqdm import tqdm
from collections import defaultdict
from itertools import permutations

# --- CONFIGURATION ---
BASE_MODEL_ID = "Farid59/nllb-darija-lora-model"
TEST_DATA_FILE = "test_dataset.json"
NUM_SAMPLES_TO_TEST = 10  # <-- MODIFICATION : Nombre d'exemples à tester
# ---------------------

def evaluate_base_model_performance():
    """
    Charge le modèle de base NLLB, l'évalue sur un sous-ensemble du jeu de test
    et affiche les scores SacreBLEU pour chaque direction de traduction.
    """
    print(f"🚀 Évaluation du modèle de base : {BASE_MODEL_ID}")
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Utilisation du périphérique : {device.upper()}")

    print("...Chargement du modèle et du tokenizer (peut prendre un moment)...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
    model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL_ID).to(device)
    model.eval()
    print("✅ Modèle et tokenizer chargés.")
    
    print(f"📂 Chargement du jeu de test depuis : {TEST_DATA_FILE}")
    full_test_dataset = load_dataset("json", data_files=TEST_DATA_FILE, split="train")

    # --- MODIFICATION : Sélectionner un sous-ensemble du jeu de données ---
    if NUM_SAMPLES_TO_TEST is not None and len(full_test_dataset) > NUM_SAMPLES_TO_TEST:
        print(f"✂️  Sélection des {NUM_SAMPLES_TO_TEST} premiers exemples pour un test rapide...")
        test_dataset_subset = full_test_dataset.select(range(NUM_SAMPLES_TO_TEST))
    else:
        test_dataset_subset = full_test_dataset
    
    print("...Matérialisation du jeu de données en mémoire...")
    test_data_list = list(test_dataset_subset)
    print(f"✅ {len(test_data_list)} exemples chargés pour le test.")

    results = defaultdict(lambda: {"predictions": [], "references": []})
    
    print("\n⚙️  Démarrage de l'inférence sur le jeu de test...")
    with torch.no_grad():
        for example in tqdm(test_data_list, desc="Traduction des phrases"):
            translation_data = example.get('translation', {})
            
            valid_translations = {
                lang: text 
                for lang, text in translation_data.items() if text is not None and text.strip()
            }
            
            if len(valid_translations) < 2:
                continue

            for source_lang, target_lang in permutations(valid_translations.keys(), 2):
                source_text = valid_translations[source_lang]
                target_text = valid_translations[target_lang]

                input_text = f">>{source_lang}<< {source_text}"
                inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=1024).to(device)

                generated_tokens = model.generate(
                    **inputs,
                    forced_bos_token_id=tokenizer.convert_tokens_to_ids(target_lang),
                    max_new_tokens=256
                )
                
                prediction = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
                
                direction = f"{source_lang} -> {target_lang}"
                results[direction]["predictions"].append(prediction)
                results[direction]["references"].append([target_text])

    print("✅ Inférence terminée.")

    print("\n=======================================================")
    print(f"   SCORES BLEU DU MODÈLE DE BASE ({BASE_MODEL_ID}) sur {len(test_data_list)} exemples")
    print("=======================================================")
    
    bleu_metric = load("sacrebleu")
    
    for direction, data in results.items():
        if data["predictions"]:
            score = bleu_metric.compute(predictions=data["predictions"], references=data["references"])
            print(f"  - Direction {direction.ljust(25)}: BLEU = {score['score']:.2f}")

    print("=======================================================")


if __name__ == "__main__":
    evaluate_base_model_performance()

🚀 Évaluation du modèle de base : Farid59/nllb-darija-lora-model
Utilisation du périphérique : CUDA
...Chargement du modèle et du tokenizer (peut prendre un moment)...
✅ Modèle et tokenizer chargés.
📂 Chargement du jeu de test depuis : test_dataset.json
✂️  Sélection des 10 premiers exemples pour un test rapide...
...Matérialisation du jeu de données en mémoire...
✅ 10 exemples chargés pour le test.

⚙️  Démarrage de l'inférence sur le jeu de test...


Traduction des phrases: 100%|██████████| 10/10 [06:03<00:00, 36.40s/it]


✅ Inférence terminée.

   SCORES BLEU DU MODÈLE DE BASE (Farid59/nllb-darija-lora-model) sur 10 exemples
  - Direction ary_Arab -> eng_Latn     : BLEU = 15.02
  - Direction eng_Latn -> ary_Arab     : BLEU = 0.00
  - Direction fra_Latn -> ary_Arab     : BLEU = 0.00
  - Direction ary_Arab -> fra_Latn     : BLEU = 35.13
